# 01 — Chuẩn bị đầu vào SVM RGB224

Notebook lập manifest train/dev theo sáu mệnh giá VND, kiểm tra khả năng đọc ảnh và ghi cấu hình cố định. Các notebook sau dùng đúng thứ tự ảnh này khi lưu feature và nhãn.


In [ ]:
# KAGGLE_INPUT_SETUP: tìm code và DATA trong các Dataset đã gắn.
from pathlib import Path
import os

kaggle_input = Path('/kaggle/input')
dataset_roots = []
if kaggle_input.is_dir():
    dataset_roots.extend(kaggle_input.glob('datasets/*/*'))
    dataset_roots.extend(kaggle_input.glob('*'))
dataset_roots = [p for p in dataset_roots if p.is_dir()]

project_hint = os.environ.get('CS231_PROJECT_ROOT')
project_candidates = ([Path(project_hint)] if project_hint else
                      [Path.cwd(), *Path.cwd().parents, *dataset_roots])
project_matches = sorted({p.resolve() for p in project_candidates
                          if (p / 'src/SVM/pipeline.py').is_file()})
if len(project_matches) != 1:
    raise FileNotFoundError(
        'Cần gắn đúng Dataset code chứa src/SVM/pipeline.py; '
        'nếu có nhiều bản, đặt CS231_PROJECT_ROOT. '
        f'Tìm thấy: {project_matches}'
    )
project = project_matches[0]

data_hint = os.environ.get('CS231_DATA_ROOT')
data_candidates = ([Path(data_hint)] if data_hint else
                   [project / 'DATA', *(p / 'DATA' for p in dataset_roots)])
data_matches = sorted({p.resolve() for p in data_candidates
                       if (p / 'train/10000').is_dir()})
if len(data_matches) != 1:
    raise FileNotFoundError(
        'Cần gắn đúng Dataset DATA chứa DATA/train/10000; '
        'nếu có nhiều bản, đặt CS231_DATA_ROOT. '
        f'Tìm thấy: {data_matches}'
    )
data = data_matches[0]

os.environ['CS231_PROJECT_ROOT'] = str(project)
os.environ['CS231_DATA_ROOT'] = str(data)
print('Code:', project, (project / 'src/SVM/pipeline.py').is_file())
print('DATA:', data, (data / 'train/10000').is_dir())


In [ ]:
from pathlib import Path
import os, sys

# Có thể chỉ rõ ba đường dẫn bằng biến môi trường khi gắn repository/dataset trên Kaggle.
root_hint = os.environ.get('CS231_PROJECT_ROOT')
root_candidates = [Path(root_hint)] if root_hint else [Path.cwd(), *Path.cwd().parents]
if Path('/kaggle/input').is_dir():
    root_candidates += list(Path('/kaggle/input').glob('*'))
    root_candidates += list(Path('/kaggle/input').glob('*/*'))
PROJECT_ROOT = next((p.resolve() for p in root_candidates
                     if (p / 'src/SVM/pipeline.py').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Không tìm thấy repository. Đặt CS231_PROJECT_ROOT.')
sys.path.insert(0, str(PROJECT_ROOT))

data_hint = os.environ.get('CS231_DATA_ROOT')
data_candidates = [Path(data_hint)] if data_hint else [PROJECT_ROOT / 'DATA']
if not data_hint and Path('/kaggle/input').is_dir():
    data_candidates += list(Path('/kaggle/input').glob('*/DATA'))
    data_candidates += list(Path('/kaggle/input').glob('*/*/DATA'))
DATA_ROOT = next((p.resolve() for p in data_candidates
                  if (p / 'train/10000').is_dir()), None)
if DATA_ROOT is None:
    raise FileNotFoundError('Không tìm thấy DATA/train/10000. Đặt CS231_DATA_ROOT.')

result_hint = os.environ.get('CS231_SVM_RESULTS_ROOT')
if result_hint:
    RESULTS_ROOT = Path(result_hint).resolve()
elif Path('/kaggle/working').is_dir():
    RESULTS_ROOT = Path('/kaggle/working/CS231_SVM/results_224')
else:
    RESULTS_ROOT = PROJECT_ROOT / 'src/SVM/results_224'
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print('Repository:', PROJECT_ROOT)
print('DATA:', DATA_ROOT)
print('Kết quả:', RESULTS_ROOT)


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.SVM.pipeline import (
    CLASSES, FEATURE_FOLDERS, manifest_for_split, extract_matrix, l2,
    save_features, load_features, same_manifest, run_dev_grid, save_dev_result,
    save_json, hsv12, hog_raw,
)
from src.preprocessing import read_rgb224


## 1. Lập manifest và kiểm tra quy mô

Đường dẫn trong CSV tương đối với `DATA`. Thứ tự lớp theo mệnh giá tăng dần; tên ảnh được sắp trong từng lớp.


In [ ]:
INPUTS = RESULTS_ROOT / 'inputs'
INPUTS.mkdir(parents=True, exist_ok=True)
train_manifest = manifest_for_split(DATA_ROOT, 'train')
dev_manifest = manifest_for_split(DATA_ROOT, 'dev')
train_manifest.to_csv(INPUTS / 'train_manifest.csv', index=False)
dev_manifest.to_csv(INPUTS / 'dev_manifest.csv', index=False)
counts = pd.DataFrame({
    'train': train_manifest.label_vnd.value_counts().reindex(CLASSES, fill_value=0),
    'dev': dev_manifest.label_vnd.value_counts().reindex(CLASSES, fill_value=0),
})
counts.loc['Total'] = counts.sum()
counts.to_csv(INPUTS / 'image_counts.csv')
display(counts)


## 2. Kiểm tra và minh họa RGB224

Ảnh được đọc bằng OpenCV, đổi BGR sang RGB rồi resize trực tiếp về 224×224 bằng `INTER_LINEAR`. Việc đổi kích thước chỉ xảy ra trong bộ nhớ; ảnh PNG trong DATA không bị ghi đè.


In [ ]:
checks = []
for split, manifest in [('train', train_manifest), ('dev', dev_manifest)]:
    for relative_path in manifest.relative_path:
        image = read_rgb224(DATA_ROOT / relative_path)
        if image.shape != (224, 224, 3) or image.dtype != np.uint8:
            raise ValueError(f'Ảnh sai định dạng: {relative_path}')
    checks.append({'split': split, 'images': len(manifest), 'rgb224_uint8': len(manifest)})
pd.DataFrame(checks).to_csv(INPUTS / 'input_checks.csv', index=False)
display(pd.DataFrame(checks))


In [ ]:
import cv2
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for col, label in enumerate(CLASSES):
    path = DATA_ROOT / train_manifest.loc[train_manifest.label_vnd == label, 'relative_path'].iloc[0]
    original = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
    resized = read_rgb224(path)
    for row, image in enumerate([original, resized]):
        axes[row, col].imshow(image)
        axes[row, col].axis('off')
    axes[0, col].set_title(f'{label:,} VND\n{original.shape[1]}×{original.shape[0]}')
    axes[1, col].set_title('RGB 224×224')
plt.tight_layout()
plt.savefig(INPUTS / 'rgb224_examples.png', dpi=150)
plt.show()


## 3. Khóa thiết kế khảo sát

Histogram HSV12, HOG và HOG-PCA dùng đúng định nghĩa trong nhánh KNN. SVC được khảo sát bằng linear và RBF với một lưới nhỏ khai báo trước khi xem kết quả dev. Macro-F1 là tiêu chí chọn chính; MAE VND phá hòa.


In [ ]:
from src.SVM.pipeline import candidates, SPLIT_SIZES, FEATURE_ORDER
config = {
    'classes_vnd': CLASSES, 'splits': SPLIT_SIZES,
    'preprocessing': {'color': 'RGB', 'size': [224, 224], 'dtype': 'uint8',
                      'resize': 'direct', 'library': 'opencv',
                      'interpolation': 'INTER_LINEAR', 'augmentation': False},
    'hsv': {'bins': [12, 12, 12], 'ranges': [0, 180, 0, 256, 0, 256], 'norm': 'L2'},
    'hog': {'grayscale': True, 'orientations': 9, 'pixels_per_cell': [16, 16],
            'cells_per_block': [2, 2], 'block_norm': 'L2-Hys',
            'transform_sqrt': False, 'dimensions_before_pca': 6084},
    'pca': {'n_components': 0.95, 'svd_solver': 'full', 'whiten': False,
            'fit_split': 'train', 'norm_after_transform': 'L2'},
    'fusion': 'concatenate separately L2-normalized HSV12 and HOG branch',
    'feature_order': FEATURE_ORDER, 'svm_candidates': candidates(),
    'selection': ['max unrounded macro_f1', 'min unrounded mae_vnd',
                  'feature order then candidate order'],
    'test_history': 'Previously inspected in earlier experiments; not an untouched holdout.',
}
save_json(INPUTS / 'experiment_config.json', config)
print('Số cấu hình mỗi biến thể:', len(candidates()))
print('Tổng cấu hình:', len(candidates()) * len(FEATURE_ORDER))
